# 02 — Data Cleaning

**Objetivo:** Corrigir todos os problemas identificados no profiling e construir o dataframe master que será usado nas análises.

**Transformações aplicadas:**
- Conversão de tipos (datas, valores monetários, horas)
- Correção de nomes de colunas
- Criação de colunas derivadas (dia da semana, mês, flag de entrega com problema)
- Join entre pedidos, clientes e motoristas

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loader import load_orders, load_customers, load_products, load_drivers, load_order_items
from src.preprocessing import clean_orders, clean_products, clean_drivers, build_master

pd.set_option("display.max_columns", None)

## 1. Limpeza individual de cada tabela

In [2]:
orders   = clean_orders(load_orders())
products = clean_products(load_products())
drivers  = clean_drivers(load_drivers())
customers = load_customers()
order_items = load_order_items()

print("orders:", orders.shape)
print("products:", products.shape)
print("drivers:", drivers.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)

orders: (10000, 12)
products: (314, 4)
drivers: (1247, 4)
customers: (1239, 3)
order_items: (1662, 2)


## 2. Verificação dos tipos após limpeza

In [3]:
print("=== ORDERS ===")
print(orders.dtypes)
print("\nAmostra:")
display(orders[["date", "order_amount", "delivery_hour", "day_of_week", "month", "has_missing"]].head())

=== ORDERS ===
date               datetime64[ns]
order_id                   object
order_amount              float64
region                     object
items_delivered             int64
items_missing               int64
delivery_hour               int32
driver_id                  object
customer_id                object
day_of_week                object
month                      object
has_missing                  bool
dtype: object

Amostra:


,date,order_amount,delivery_hour,day_of_week,month,has_missing
0,2023-01-01,1095.54,8,Sunday,2023-01,True
1,2023-01-01,659.11,9,Sunday,2023-01,True
2,2023-01-01,251.45,10,Sunday,2023-01,True
3,2023-01-01,598.83,9,Sunday,2023-01,True
4,2023-01-01,27.18,10,Sunday,2023-01,True


In [4]:
print("=== PRODUCTS ===")
print(products.dtypes)
display(products.head(3))

=== PRODUCTS ===
product_id       object
product_name     object
category         object
price           float64
dtype: object


,product_id,product_name,category,price
0,PWPX0982761090982,Kellogg's Frosties,Supermarket,12.53
1,PWPX0982761090983,Uncured Bacon,Supermarket,4.67
2,PWPX0982761090984,Whole Milk,Supermarket,9.95


## 3. Construção do dataframe master

In [5]:
master = build_master(orders, customers, drivers)

print(f"Master dataframe: {master.shape[0]:,} linhas | {master.shape[1]} colunas")
display(master.head())

Master dataframe: 10,000 linhas | 17 colunas

,date,order_id,order_amount,region,items_delivered,items_missing,delivery_hour,driver_id,customer_id,day_of_week,month,has_missing,customer_name,customer_age,driver_name,age,trips
0,2023-01-01,c9da15aa-be24-4871-92a3-dfa7746fff69,1095.54,Winter Park,10,1,8,WDID10627,WCID5031,Sunday,2023-01,True,John Cooper,68,Jeremy Ramos,64,44
1,2023-01-01,ccacc183-09f8-4fd5-af35-009d18656326,659.11,Altamonte Springs,11,1,9,WDID10533,WCID5794,Sunday,2023-01,True,Christina Watts,30,Michelle Robinson,41,45
2,2023-01-01,f4e1d30b-c3d1-413f-99b8-93c0b46d68bf,251.45,Winter Park,18,1,10,WDID10559,WCID5599,Sunday,2023-01,True,Natalie Fisher,26,Thomas White,30,67
3,2023-01-01,993d31f4-9358-41f0-a371-0021e55cef5d,598.83,Altamonte Springs,12,1,9,WDID10622,WCID5005,Sunday,2023-01,True,Dawn Tucker,85,Daniel Newman,61,34
4,2023-01-01,3e0a8f1b-3cd6-4d64-90e3-6b38dc368925,27.18,Clermont,3,1,10,WDID10654,WCID5114,Sunday,2023-01,True,Jasmine Lee,57,Chris Ryan,51,31


## 4. Validação final

In [6]:
print("Nulos no master:")
nulls = master.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "Nenhum valor nulo.")

print("\nIntervalo de datas:")
print(f"  De: {master['date'].min().date()}")
print(f"  Até: {master['date'].max().date()}")

print("\nReceita total:")
print(f"  R$ {master['order_amount'].sum():,.2f}")

print("\nTaxa geral de itens faltando:")
print(f"  {master['has_missing'].mean()*100:.1f}% dos pedidos")

Nulos no master:


Nenhum valor nulo.

Intervalo de datas:
  De: 2023-01-01
  Até: 2023-12-31

Receita total:
  R$ 2,833,022.38

Taxa geral de itens faltando:
  15.0% dos pedidos


## 5. Exportar dados limpos

In [7]:
import os
os.makedirs("../data/processed", exist_ok=True)

master.to_parquet("../data/processed/master.parquet", index=False)
products.to_parquet("../data/processed/products.parquet", index=False)
order_items.to_parquet("../data/processed/order_items.parquet", index=False)

print("Dados exportados para data/processed/")

Dados exportados para data/processed/
